# 🎙️ VoiceCloner — Viterbox TTS trên Colab

Repo: https://github.com/kanazawahere/VoiceCloner

**Tính năng:**
- ✅ Python 3.12 compatible
- ✅ Upload file TXT để đọc
- ✅ Điều chỉnh tốc độ và cao độ
- ✅ Zero-shot voice cloning
- ✅ Cache model vào Drive

In [ ]:
# ✅ BƯỚC 1: Trỏ HuggingFace cache về Drive
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
os.makedirs(DRIVE_HF_CACHE, exist_ok=True)

os.environ['HF_HOME'] = DRIVE_HF_CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE

print(f"✅ HF cache: {DRIVE_HF_CACHE}")

In [ ]:
# ✅ BƯỚC 2: Cài uv
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os
current_path = os.environ.get('PATH', '')
new_path = f"/root/.local/bin:{current_path}"
os.environ['PATH'] = new_path
%env PATH={new_path}
!uv --version

In [ ]:
# ✅ BƯỚC 3: Clone repo
import shutil
if os.path.exists('/content/VoiceCloner'):
    shutil.rmtree('/content/VoiceCloner')

!git clone https://github.com/kanazawahere/VoiceCloner.git \
    /content/VoiceCloner --depth=1 --quiet
%cd /content/VoiceCloner
print('✅ Clone xong')

In [ ]:
# ✅ BƯỚC 4: Cài packages
import subprocess
cuda_ver = subprocess.getoutput(
    r"nvcc --version | grep 'release' | sed 's/.*release //' | sed 's/,.*//' | sed 's/\.//'"
).strip()
if not cuda_ver.isdigit():
    cuda_ver = "121"
torch_index = f"https://download.pytorch.org/whl/cu{cuda_ver}"
print(f"CUDA: cu{cuda_ver}")

!uv pip install --system torch torchvision torchaudio \
    --index-url {torch_index} --quiet
!uv pip install --system -r requirements.txt --quiet
!uv pip install --system -e . --quiet

print("✅ Cài xong")
print("⚠️ QUAN TRỌNG: Chạy cell tiếp theo để restart runtime!")

In [ ]:
# ✅ BƯỚC 4.5: Restart runtime
import os
os.kill(os.getpid(), 9)

In [ ]:
# ✅ BƯỚC 5: Kiểm tra môi trường sau restart
import os

if 'HF_HOME' not in os.environ:
    DRIVE_HF_CACHE = "/content/drive/MyDrive/viterbox_hf_cache"
    os.environ['HF_HOME'] = DRIVE_HF_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_HF_CACHE

if not os.path.exists('/content/VoiceCloner'):
    print("⚠️ Chạy lại từ BƯỚC 3!")
else:
    %cd /content/VoiceCloner
    print("✅ Sẵn sàng")

In [ ]:
# ✅ BƯỚC 6: Load model
import torch
from viterbox import Viterbox

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tts = Viterbox.from_pretrained(device)
print("✅ Model load xong!")

---
## 📄 Upload file TXT để đọc

In [ ]:
# ✅ BƯỚC 7: Upload file TXT
from google.colab import files

print("📤 Upload file TXT:")
uploaded = files.upload()
txt_file = list(uploaded.keys())[0]

with open(txt_file, 'r', encoding='utf-8') as f:
    text_content = f.read()

print(f"\n✅ Đã đọc {len(text_content)} ký tự")
print(f"Preview: {text_content[:200]}...")

---
## 🎤 Upload giọng mẫu (tùy chọn)

Bỏ qua nếu muốn dùng giọng mặc định

In [ ]:
# ✅ BƯỚC 8: Upload giọng mẫu (tùy chọn)
from google.colab import files

USE_VOICE_CLONING = True  # ← Đổi thành False nếu dùng giọng mặc định

if USE_VOICE_CLONING:
    print("📤 Upload file WAV giọng mẫu (3-10 giây):")
    uploaded = files.upload()
    ref_audio = list(uploaded.keys())[0]
    print(f"✅ Đã upload: {ref_audio}")
else:
    ref_audio = None
    print("✅ Dùng giọng mặc định")

---
## 🎛️ Điều chỉnh tốc độ và cao độ

In [ ]:
# ✅ BƯỚC 9: Generate audio với điều chỉnh
import librosa
import soundfile as sf
from IPython.display import Audio, display
import numpy as np

# ========== CÀI ĐẶT ==========
SPEED = 1.0      # Tốc độ: 0.5 = chậm 2x, 1.0 = bình thường, 1.5 = nhanh 1.5x
PITCH_SHIFT = 0  # Cao độ: -5 = thấp hơn, 0 = bình thường, +5 = cao hơn
# ==============================

print("🎙️ Đang generate audio...")
audio = tts.generate(
    text=text_content,
    language="vi",
    audio_prompt=ref_audio,
    sentence_pause_ms=500
)

# Convert to numpy
audio_np = audio.squeeze().cpu().numpy()
sr = 24000

# Điều chỉnh tốc độ
if SPEED != 1.0:
    print(f"⚡ Điều chỉnh tốc độ: {SPEED}x")
    audio_np = librosa.effects.time_stretch(audio_np, rate=SPEED)

# Điều chỉnh cao độ
if PITCH_SHIFT != 0:
    print(f"🎵 Điều chỉnh cao độ: {PITCH_SHIFT:+d} semitones")
    audio_np = librosa.effects.pitch_shift(audio_np, sr=sr, n_steps=PITCH_SHIFT)

# Lưu file
output_file = "/content/output_final.wav"
sf.write(output_file, audio_np, sr)

print(f"\n✅ Hoàn tất!")
print(f"Thời lượng: {len(audio_np)/sr:.1f}s")
display(Audio(output_file))

---
## 💾 Lưu vào Drive

In [ ]:
# ✅ BƯỚC 10: Lưu output vào Drive
import shutil
import os
from datetime import datetime

OUTPUT_DIR = "/content/drive/MyDrive/viterbox_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Tạo tên file với timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_name = f"output_{timestamp}.wav"

shutil.copy("/content/output_final.wav", f"{OUTPUT_DIR}/{output_name}")
print(f"✅ Đã lưu: {output_name}")
print(f"📁 Thư mục: {OUTPUT_DIR}")

---
## 📋 Hướng dẫn sử dụng

### Điều chỉnh tốc độ và cao độ

Sửa trong **BƯỚC 9**:

```python
SPEED = 1.0      # Tốc độ
  0.5  = chậm 2x
  0.75 = chậm 1.3x
  1.0  = bình thường
  1.25 = nhanh 1.25x
  1.5  = nhanh 1.5x

PITCH_SHIFT = 0  # Cao độ (semitones)
  -5 = thấp hơn (giọng nam)
   0 = bình thường
  +5 = cao hơn (giọng nữ/trẻ em)
```

### Tips
- File TXT nên mã hóa UTF-8
- Giọng mẫu: 3-10 giây, sạch, không nhiễu
- Tốc độ quá nhanh (>1.5x) có thể làm giảm chất lượng
- Cao độ nên điều chỉnh trong khoảng -5 đến +5